# CASPER Training Pipeline

**CASPER: Continuous Action Space Preference Elicitation via Reinforcement**

Complete training pipeline:
1. Concept model reward signal (Accuracy using ConceptEmbeddingModel)
2. LSTM+Attention state encoder (Concept model style)
3. RL actor supervised pretraining (Reddit data)
4. End-to-end RL training with per-turn accuracy rewards
5. Baseline evaluation

## Key Architecture Changes
- **State Encoder**: LSTM+Attention over (item, rating) sequences (vs. SBERT text encoding)
- **Reward Signal**: Concept model predicts all items, reward = Accuracy(t) - Accuracy(t-1)
- **Action Space**: Continuous concept embeddings (384-dim SentenceBERT)

## Reward Signal: Concept Model Accuracy
- SNR = 1.69 (vs NDCG SNR = 0.81) — 2x more learnable
- Perfectly monotonic (20/20 steps improve)
- Dense: computed over ALL rated items (not just top-10)
- Fast, free, deterministic (no LLM API calls)

## 1. Environment Setup

In [ ]:
import sys
import os
import random
from pathlib import Path
import torch
import numpy as np
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
load_dotenv(PROJECT_ROOT / '.env')

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError("OPENAI_API_KEY not found in .env file")

# Set all random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Device: {device}")
print(f"Random seeds set: 42")

## 2. Load Configuration

In [ ]:
import yaml

with open(PROJECT_ROOT / 'config' / 'config.yaml', 'r') as f:
    config = yaml.safe_load(f)

PRETRAIN_EPOCHS = config['training_data']['supervised']['epochs']
RL_EPISODES = config['training_data']['reinforcement']['num_episodes']
MAX_TURNS = config['environment']['max_turns']
DATA_DIR = PROJECT_ROOT / 'data' / 'movielens'

print(f"Configuration:")
print(f"  Pretraining epochs: {PRETRAIN_EPOCHS}")
print(f"  RL episodes: {RL_EPISODES}")
print(f"  Max turns: {MAX_TURNS}")
print(f"  Model: {config['models']['question_generator']['model_name']}")

## 3. Initialize Components

In [ ]:
from casper.data.movielens_loader import MovieLensLoader
from casper.models.embedding_space import SentenceBERTEmbeddingSpace
from casper.models.rl_actor_critic import EmbeddingActorCritic, LSTMAttentionStateEncoder
from casper.models.two_tower_recommender import TwoTowerRecommender, MovieCatalog, RecommenderTrainer
from casper.models.preference_extractor import PreferenceExtractor
from casper.agents.user_simulator import UserSimulator
from casper.agents.casper_agent import CASPERAgent
from casper.baselines.random_agent import RandomAgent
from casper.baselines.pure_llm_agent import PureLLMAgent
from casper.evaluation.conversation_evaluator import ConversationalEvaluator
from casper.evaluation import metrics

loader = MovieLensLoader(data_path=str(DATA_DIR), min_ratings=20)
train_users, test_users = loader.load_data(test_split=0.3)

shared_embedding_space = SentenceBERTEmbeddingSpace(str(DATA_DIR))

casper_agent = CASPERAgent(
    movielens_data_path=str(DATA_DIR),
    load_recommender=True,
    embedding_space=shared_embedding_space
)

# Enable LSTM+attention state encoding (Concept model style)
# This encodes each (item, rating) pair individually instead of full text
casper_agent.set_state_encoder_type('lstm')

user_sim = UserSimulator(
    movielens_data_path=str(DATA_DIR),
    min_ratings=config['models']['user_simulator']['min_ratings'],
    max_profiles=config['models']['user_simulator']['max_profiles']
)

random_agent = RandomAgent(
    movielens_data_path=str(DATA_DIR),
    embedding_space=shared_embedding_space
)

llm_agent = PureLLMAgent(
    movielens_data_path=str(DATA_DIR),
    model_name='gpt-4o-mini',
    embedding_space=shared_embedding_space
)

movie_catalog = MovieCatalog(str(DATA_DIR), encoder=casper_agent.encoder)
shared_recommender_model = TwoTowerRecommender(state_dim=384, embedding_dim=128)
shared_recommender = RecommenderTrainer(
    recommender=shared_recommender_model,
    movie_catalog=movie_catalog,
    encoder=casper_agent.encoder
)

shared_pref_extractor = PreferenceExtractor()
evaluator = ConversationalEvaluator(holdout_ratio=0.3, min_rating_threshold=4.0)

print(f"Train users: {len(train_users)}, Test users: {len(test_users)}")
print(f"State encoder type: {casper_agent.state_encoder_type}")

## 4. Concept Model Reward Setup

Instead of using an LLM for recommendations, we use the saved ConceptEmbeddingModel
(SBERT+LSTM+Attention, trained on MovieLens) to predict user preferences directly.

**Reward = Accuracy(t) - Accuracy(t-1)**

Where accuracy = fraction of rated items correctly predicted (pred > 0.5 matches ground truth).

This approach:
1. Uses the concept model's learned movie embeddings (384-dim SBERT)
2. Accuracy has SNR=1.69 — 2x stronger signal than NDCG (SNR=0.81)
3. Perfectly monotonic: every revealed preference improves accuracy
4. Dense signal: computed over ALL rated items (not just top-10 ranking)
5. No API calls: fast, free, deterministic

In [ ]:
from casper.evaluation.concept_reward import ConceptModelRewardCalculator
from casper.training.episode_runner import EpisodeRunner

# Initialize Concept Model reward calculator
reward_calc = ConceptModelRewardCalculator(data_path=str(DATA_DIR), top_n_movies=100)

# Filter users to only those with enough rated movies in the pool
# For accuracy, we count ALL rated movies (liked + disliked), not just liked
MIN_EVAL_TARGETS = 10  # At least 10 rated movies in TOP_100

print(f"Filtering users by eval targets (min={MIN_EVAL_TARGETS})...")
print(f"  Before: {len(user_sim.user_profiles)} users")

user_sim.user_profiles = reward_calc.filter_users_by_eval_targets(
    user_sim.user_profiles, 
    min_eval_targets=MIN_EVAL_TARGETS
)

print(f"  After: {len(user_sim.user_profiles)} users")

eval_counts = [reward_calc.count_eval_targets(p) for p in user_sim.user_profiles]
if eval_counts:
    print(f"  Eval targets: min={min(eval_counts)}, max={max(eval_counts)}, mean={sum(eval_counts)/len(eval_counts):.1f}")

# Initialize episode runner
# Key changes from LLM reward:
# - use_baseline=True: Accuracy baseline ~0.49 (near chance), so deltas are small and uniform
# - exclude_mentioned=False: Concept model accuracy doesn't benefit from filtering
# - new_pref_bonus=0.05: Bonus per new liked/disliked preference discovered
# - not_seen_penalty=0.05: Penalty for "haven't seen that" responses
episode_runner = EpisodeRunner(
    reward_calculator=reward_calc,
    user_simulator=user_sim,
    max_turns=MAX_TURNS,
    holdout_ratio=0.0,
    exclude_mentioned=False,  # Concept model accuracy doesn't need filtering
    # Reward shaping parameters
    ndcg_weight=1.0,         # Weight for accuracy improvement (param name kept for compat)
    new_pref_bonus=0.05,     # Bonus per new liked/disliked preference
    not_seen_penalty=0.05,   # Penalty per "not seen" response
    use_baseline=True         # Start from baseline accuracy (~0.49), reward improvement
)

print(f"\nConcept model reward calculator initialized")
print(f"  Movie pool: {len(reward_calc.movie_list)} movies (eval)")
print(f"  Model items: {reward_calc.n_items} (full model)")
print(f"\nEpisode runner initialized")
print(f"  Max turns: {MAX_TURNS}")
print(f"  Exclude mentioned: {episode_runner.exclude_mentioned}")
print(f"  Use baseline: {episode_runner.use_baseline}")
print(f"  Reward shaping:")
print(f"    - Accuracy weight: {episode_runner.ndcg_weight}")
print(f"    - New pref bonus: {episode_runner.new_pref_bonus}")
print(f"    - Not seen penalty: {episode_runner.not_seen_penalty}")

## 5. Download Reddit Training Data

Download Reddit conversations for supervised actor pretraining.

**Purpose**: Train RL actor to predict good follow-up concepts from user preferences
**Subreddits**: r/MovieSuggestions, r/movies, r/TrueFilm
**Format**: User post → Expert response pairs

In [ ]:
from casper.data.reddit_scraper import RedditScraper
import json
import time

reddit_data_dir = PROJECT_ROOT / 'data' / 'reddit'
reddit_data_dir.mkdir(parents=True, exist_ok=True)

existing_files = list(reddit_data_dir.glob('*_conversations.json'))
total_pairs = 0
if existing_files:
    for json_file in existing_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                pairs = json.load(f)
                total_pairs += len(pairs)
        except:
            pass

if total_pairs > 0:
    print(f"Found existing Reddit data: {total_pairs} conversation pairs in {len(existing_files)} files")
    print(f"Location: {reddit_data_dir}")
else:
    print("Downloading Reddit training data...")
    
    scraper = RedditScraper(output_dir=str(reddit_data_dir), use_semantic_filter=True)
    
    # Download conversation PAIRS (user post + expert response)
    # This is what we need for pretraining the actor
    print(f"\nScraping r/MovieSuggestions (target: 500 conversation pairs)...")
    try:
        pairs = scraper.scrape_conversation_pairs(
            subreddit='MovieSuggestions',
            max_pairs=500
        )
        
        if pairs:
            output_file = reddit_data_dir / 'moviesuggestions_conversations.json'
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(pairs, f, indent=2, ensure_ascii=False)
            
            print(f"  Scraped {len(pairs)} conversation pairs")
            print(f"  Saved to: {output_file}")
        else:
            print(f"  No conversation pairs retrieved")
        
    except Exception as e:
        print(f"  Error scraping: {e}")
        import traceback
        traceback.print_exc()
    
    if len(pairs) > 0:
        print(f"\nReddit data download complete!")
        print(f"  Total conversation pairs: {len(pairs)}")
        print(f"  Saved to: {reddit_data_dir}")
    else:
        print(f"\nWARNING: No Reddit data was downloaded!")
        print(f"  Supervised pretraining will be skipped.")

## 5b. RL Actor Supervised Pretraining

Pretrain actor to predict embeddings in correct semantic space using Reddit conversation data.

**Goal:** Learn (state → concept embedding) mapping BEFORE RL training
**Data:** Reddit conversations (r/MovieSuggestions, r/movies)
**Loss:** MSE between predicted and target concept embeddings

**LSTM Mode:** When using LSTM state encoder, we also train the state encoder jointly.
The state is built from preference sequences: [(item1, liked), (item2, disliked), ...]

In [ ]:
from casper.data.reddit_loader import RedditDataLoader
from casper.data.batch_reddit_preprocessor import BatchRedditPreprocessor
from casper.models.rl_actor_critic import create_rating_batch
from tqdm import tqdm
import json

pretrain_checkpoint = CHECKPOINT_DIR / 'actor_pretrained_lstm.pt' if casper_agent.state_encoder_type == 'lstm' else CHECKPOINT_DIR / 'actor_pretrained.pt'

if pretrain_checkpoint.exists():
    print(f"Loading pretrained model from {pretrain_checkpoint}")
    checkpoint = torch.load(pretrain_checkpoint)
    casper_agent.rl_agent.actor.load_state_dict(checkpoint['actor'])
    if 'state_encoder' in checkpoint and casper_agent.rl_agent.state_encoder is not None:
        casper_agent.rl_agent.state_encoder.load_state_dict(checkpoint['state_encoder'])
        print(f"State encoder loaded")
    print(f"Model loaded from pretrained checkpoint")
else:
    if not reddit_data_dir.exists():
        raise RuntimeError(f"Reddit data directory not found: {reddit_data_dir}")
    
    # Check if we have preprocessed data (with extracted concepts)
    preprocessed_files = list(reddit_data_dir.glob('*_processed.json'))
    
    if not preprocessed_files:
        print(f"No preprocessed Reddit data found. Running batch LLM extraction...")
        print(f"This will extract preferences and concepts using batched async LLM calls.")
        
        # Run batch preprocessing
        preprocessor = BatchRedditPreprocessor(batch_size=50)
        
        # Process all conversation files
        for conv_file in reddit_data_dir.glob('*_conversations.json'):
            output_file = conv_file.parent / f"{conv_file.stem}_processed.json"
            print(f"\nProcessing {conv_file.name}...")
            preprocessor.process_reddit_file(conv_file, output_file)
        
        preprocessed_files = list(reddit_data_dir.glob('*_processed.json'))
    
    if not preprocessed_files:
        raise RuntimeError(f"No preprocessed data found after processing!")
    
    print(f"\nLoading preprocessed Reddit data...")
    
    # Load all preprocessed data
    all_data = []
    for json_file in preprocessed_files:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            all_data.extend(data)
    
    print(f"Loaded {len(all_data)} preprocessed conversation pairs")
    print(f"Creating training examples (mode: {casper_agent.state_encoder_type})...")
    
    try:
        training_examples = []
        
        for pair in tqdm(all_data, desc="Processing pairs", unit="pair"):
            user_prefs = pair.get('user_preferences', {})
            response_concepts = pair.get('response_concepts', [])
            
            # Skip if no concepts extracted
            if not response_concepts:
                continue
            
            # Skip if user_preferences is empty
            if not any(user_prefs.get(k, []) for k in ['liked', 'neutral', 'disliked']):
                continue
            
            # Build preference sequence for LSTM mode
            pref_sequence = []
            for item in user_prefs.get('liked', []):
                pref_sequence.append((item, 'liked'))
            for item in user_prefs.get('disliked', []):
                pref_sequence.append((item, 'disliked'))
            
            if not pref_sequence:
                continue
            
            # Create training example for each concept
            for concept in response_concepts:
                concept_emb = casper_agent.embedding_space.get_embedding(concept)
                if concept_emb is not None:
                    training_examples.append({
                        'pref_sequence': pref_sequence,  # For LSTM mode
                        'user_prefs': user_prefs,  # For SBERT mode fallback
                        'concept': concept,
                        'target_embedding': concept_emb
                    })
        
        print(f"\nCreated {len(training_examples)} training examples")
        
        if len(training_examples) == 0:
            print("\nWARNING: No valid training examples created!")
            print("  Skipping supervised pretraining - model will train from scratch during RL.")
        else:
            # Prepare data based on encoder type
            if casper_agent.state_encoder_type == 'lstm':
                print(f"Preparing LSTM sequence data...")
                
                # Pre-encode all unique items
                all_items = set()
                for ex in training_examples:
                    for item, _ in ex['pref_sequence']:
                        all_items.add(item)
                
                print(f"  Encoding {len(all_items)} unique items...")
                item_list = list(all_items)
                item_embeddings_all = casper_agent.encoder.encode(
                    item_list, convert_to_numpy=True, show_progress_bar=True, batch_size=64
                )
                item_emb_map = {item: emb for item, emb in zip(item_list, item_embeddings_all)}
                
                # Build sequence data
                sequences = []
                for ex in training_examples:
                    items = [item for item, _ in ex['pref_sequence']]
                    ratings = [rating for _, rating in ex['pref_sequence']]
                    item_embs = np.array([item_emb_map[item] for item in items])
                    sequences.append({
                        'item_embeddings': item_embs,
                        'ratings': ratings,
                        'target': ex['target_embedding']
                    })
                
                # Train/val split
                val_split = 0.1
                n_val = int(len(sequences) * val_split)
                indices = np.random.permutation(len(sequences))
                train_indices = indices[:-n_val] if n_val > 0 else indices
                val_indices = indices[-n_val:] if n_val > 0 else []
                
                print(f"  Training examples: {len(train_indices)}")
                print(f"  Validation examples: {len(val_indices)}")
                
                # Training loop for LSTM mode
                print(f"\nTraining actor + state encoder for {PRETRAIN_EPOCHS} epochs...")
                
                # Get all trainable parameters
                params = list(casper_agent.rl_agent.actor.parameters())
                if casper_agent.rl_agent.state_encoder is not None:
                    params.extend(casper_agent.rl_agent.state_encoder.parameters())
                optimizer = torch.optim.Adam(params, lr=0.001)
                
                batch_size = 32
                best_val_loss = float('inf')
                best_epoch = 0
                
                for epoch in tqdm(range(PRETRAIN_EPOCHS), desc="Pretraining epochs", unit="epoch"):
                    # Training phase
                    casper_agent.rl_agent.actor.train()
                    if casper_agent.rl_agent.state_encoder is not None:
                        casper_agent.rl_agent.state_encoder.train()
                    
                    epoch_loss = 0
                    num_batches = 0
                    batch_perm = np.random.permutation(len(train_indices))
                    
                    for i in range(0, len(batch_perm), batch_size):
                        batch_idx = batch_perm[i:i+batch_size]
                        
                        # Process each example in batch (variable sequence lengths)
                        batch_states = []
                        batch_targets = []
                        
                        for idx in batch_idx:
                            seq = sequences[train_indices[idx]]
                            item_embs = torch.FloatTensor(seq['item_embeddings']).unsqueeze(0)
                            rating_oh = create_rating_batch(seq['ratings']).unsqueeze(0)
                            mask = torch.ones(1, len(seq['ratings']), dtype=torch.bool)
                            
                            state = casper_agent.rl_agent.state_encoder(item_embs, rating_oh, mask)
                            batch_states.append(state.squeeze(0))
                            batch_targets.append(torch.FloatTensor(seq['target']))
                        
                        states_tensor = torch.stack(batch_states)
                        targets_tensor = torch.stack(batch_targets)
                        
                        predicted = casper_agent.rl_agent.actor(states_tensor)
                        
                        predicted_norm = torch.nn.functional.normalize(predicted, dim=1)
                        targets_norm = torch.nn.functional.normalize(targets_tensor, dim=1)
                        
                        loss = torch.nn.functional.mse_loss(predicted_norm, targets_norm)
                        
                        optimizer.zero_grad()
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(params, 1.0)
                        optimizer.step()
                        
                        epoch_loss += loss.item()
                        num_batches += 1
                    
                    avg_train_loss = epoch_loss / num_batches if num_batches > 0 else 0
                    
                    # Validation phase
                    casper_agent.rl_agent.actor.eval()
                    if casper_agent.rl_agent.state_encoder is not None:
                        casper_agent.rl_agent.state_encoder.eval()
                    
                    val_loss = 0
                    val_cos_sim = 0
                    val_batches = 0
                    
                    if len(val_indices) > 0:
                        with torch.no_grad():
                            for i in range(0, len(val_indices), batch_size):
                                batch_idx = val_indices[i:i+batch_size]
                                
                                batch_states = []
                                batch_targets = []
                                
                                for idx in batch_idx:
                                    seq = sequences[idx]
                                    item_embs = torch.FloatTensor(seq['item_embeddings']).unsqueeze(0)
                                    rating_oh = create_rating_batch(seq['ratings']).unsqueeze(0)
                                    mask = torch.ones(1, len(seq['ratings']), dtype=torch.bool)
                                    
                                    state = casper_agent.rl_agent.state_encoder(item_embs, rating_oh, mask)
                                    batch_states.append(state.squeeze(0))
                                    batch_targets.append(torch.FloatTensor(seq['target']))
                                
                                states_tensor = torch.stack(batch_states)
                                targets_tensor = torch.stack(batch_targets)
                                
                                predicted = casper_agent.rl_agent.actor(states_tensor)
                                
                                predicted_norm = torch.nn.functional.normalize(predicted, dim=1)
                                targets_norm = torch.nn.functional.normalize(targets_tensor, dim=1)
                                
                                batch_loss = torch.nn.functional.mse_loss(predicted_norm, targets_norm)
                                val_loss += batch_loss.item()
                                
                                cos_sim = torch.nn.functional.cosine_similarity(predicted_norm, targets_norm).mean()
                                val_cos_sim += cos_sim.item()
                                val_batches += 1
                    
                    avg_val_loss = val_loss / val_batches if val_batches > 0 else avg_train_loss
                    avg_val_cos_sim = val_cos_sim / val_batches if val_batches > 0 else 0
                    
                    best_marker = ""
                    if avg_val_loss < best_val_loss:
                        best_val_loss = avg_val_loss
                        best_epoch = epoch + 1
                        best_marker = " *"
                        
                        checkpoint = {
                            'actor': casper_agent.rl_agent.actor.state_dict(),
                            'epoch': epoch + 1,
                            'train_loss': avg_train_loss,
                            'val_loss': avg_val_loss,
                            'val_cos_sim': avg_val_cos_sim,
                            'state_encoder_type': 'lstm',
                            'num_examples': len(train_indices)
                        }
                        if casper_agent.rl_agent.state_encoder is not None:
                            checkpoint['state_encoder'] = casper_agent.rl_agent.state_encoder.state_dict()
                        torch.save(checkpoint, pretrain_checkpoint)
                    
                    tqdm.write(
                        f"Epoch {epoch+1}/{PRETRAIN_EPOCHS}: "
                        f"train_loss={avg_train_loss:.4f} | "
                        f"val_loss={avg_val_loss:.4f} | "
                        f"val_cos_sim={avg_val_cos_sim:.3f}{best_marker}"
                    )
            
            else:
                # SBERT mode (original)
                print(f"Preparing SBERT text data...")
                pref_texts = [casper_agent.preference_extractor.preferences_to_text(ex['user_prefs']) 
                              for ex in training_examples]
                
                pref_embeddings = casper_agent.encoder.encode(
                    pref_texts, convert_to_numpy=True, show_progress_bar=True, batch_size=64
                )
                target_embeddings = np.array([ex['target_embedding'] for ex in training_examples])
                
                val_split = 0.1
                n_val = int(len(training_examples) * val_split)
                indices = np.random.permutation(len(training_examples))
                train_indices = indices[:-n_val]
                val_indices = indices[-n_val:]
                
                train_states = pref_embeddings[train_indices]
                train_targets = target_embeddings[train_indices]
                val_states = pref_embeddings[val_indices]
                val_targets = target_embeddings[val_indices]
                
                print(f"  Training examples: {len(train_indices)}")
                print(f"  Validation examples: {len(val_indices)}")
                
                print(f"\nTraining actor for {PRETRAIN_EPOCHS} epochs...")
                optimizer = torch.optim.Adam(casper_agent.rl_agent.actor.parameters(), lr=0.001)
                batch_size = 32
                best_val_loss = float('inf')
                best_epoch = 0
                
                for epoch in tqdm(range(PRETRAIN_EPOCHS), desc="Pretraining epochs", unit="epoch"):
                    casper_agent.rl_agent.actor.train()
                    epoch_loss = 0
                    num_batches = 0
                    batch_perm = np.random.permutation(len(train_indices))
                    
                    for i in range(0, len(batch_perm), batch_size):
                        batch_idx = batch_perm[i:i+batch_size]
                        states_tensor = torch.FloatTensor(train_states[batch_idx])
                        targets_tensor = torch.FloatTensor(train_targets[batch_idx])
                        
                        predicted = casper_agent.rl_agent.actor(states_tensor)
                        predicted_norm = torch.nn.functional.normalize(predicted, dim=1)
                        targets_norm = torch.nn.functional.normalize(targets_tensor, dim=1)
                        
                        loss = torch.nn.functional.mse_loss(predicted_norm, targets_norm)
                        optimizer.zero_grad()
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(casper_agent.rl_agent.actor.parameters(), 1.0)
                        optimizer.step()
                        
                        epoch_loss += loss.item()
                        num_batches += 1
                    
                    avg_train_loss = epoch_loss / num_batches if num_batches > 0 else 0
                    
                    casper_agent.rl_agent.actor.eval()
                    val_loss = 0
                    val_cos_sim = 0
                    val_batches = 0
                    
                    with torch.no_grad():
                        for i in range(0, len(val_indices), batch_size):
                            states_tensor = torch.FloatTensor(val_states[i:i+batch_size])
                            targets_tensor = torch.FloatTensor(val_targets[i:i+batch_size])
                            
                            predicted = casper_agent.rl_agent.actor(states_tensor)
                            predicted_norm = torch.nn.functional.normalize(predicted, dim=1)
                            targets_norm = torch.nn.functional.normalize(targets_tensor, dim=1)
                            
                            batch_loss = torch.nn.functional.mse_loss(predicted_norm, targets_norm)
                            val_loss += batch_loss.item()
                            
                            cos_sim = torch.nn.functional.cosine_similarity(predicted_norm, targets_norm).mean()
                            val_cos_sim += cos_sim.item()
                            val_batches += 1
                    
                    avg_val_loss = val_loss / val_batches if val_batches > 0 else 0
                    avg_val_cos_sim = val_cos_sim / val_batches if val_batches > 0 else 0
                    
                    best_marker = ""
                    if avg_val_loss < best_val_loss:
                        best_val_loss = avg_val_loss
                        best_epoch = epoch + 1
                        best_marker = " *"
                        
                        torch.save({
                            'actor': casper_agent.rl_agent.actor.state_dict(),
                            'epoch': epoch + 1,
                            'train_loss': avg_train_loss,
                            'val_loss': avg_val_loss,
                            'val_cos_sim': avg_val_cos_sim,
                            'state_encoder_type': 'sbert',
                            'num_examples': len(train_indices)
                        }, pretrain_checkpoint)
                    
                    tqdm.write(
                        f"Epoch {epoch+1}/{PRETRAIN_EPOCHS}: "
                        f"train_loss={avg_train_loss:.4f} | "
                        f"val_loss={avg_val_loss:.4f} | "
                        f"val_cos_sim={avg_val_cos_sim:.3f}{best_marker}"
                    )
            
            print(f"\nPretraining complete!")
            print(f"  Best epoch: {best_epoch}")
            print(f"  Best val loss: {best_val_loss:.4f}")
            print(f"  Checkpoint saved: {pretrain_checkpoint}")
        
    except Exception as e:
        print(f"\nWARNING: Supervised pretraining failed: {e}")
        print(f"  Model will train from scratch during RL.")
        import traceback
        traceback.print_exc()

## 6. Define Episode Runners

In [ ]:
# Episode runner is already initialized in cell-8
# The episode_runner.run_episode() method handles:
# - User profile preparation (ground truth vector for all rated movies in pool)
# - Baseline accuracy starts from model's zero-knowledge prediction (~0.49)
# - Per-turn reward = accuracy_delta + new_pref_bonus - not_seen_penalty
# - Capturing states/embeddings for RL training

print("Episode runner ready")
print("\nReward signal (Concept Model Accuracy):")
print("  reward = acc_weight * (Accuracy_t - Accuracy_{t-1})")
print("         + new_pref_bonus * (new_liked + new_disliked)")
print("         - not_seen_penalty * new_not_seen")
print("  - Turn 1: greeting only, reward = 0 (RL not trained)")
print("  - Turn 2+: RL predicts concepts, shaped reward calculated")
print("  - Accuracy baseline ~0.49 (near chance)")
print("  - Per-step improvement ~+0.014 (perfectly monotonic)")

# OPTIONAL: Clear replay buffer if starting fresh with new reward signal
CLEAR_REPLAY_BUFFER = True  # Set to True to clear old experiences

if CLEAR_REPLAY_BUFFER:
    print(f"\nCLEARING REPLAY BUFFER (reward function changed to accuracy)")
    casper_agent.rl_agent.clear_replay_buffer()
    print("  Network weights preserved, old experiences discarded")

## 7. RL Training Loop

End-to-end RL training with per-turn concept model accuracy rewards.

**Training:**
- Sample user from training set
- Run episode: agent asks questions, user simulator responds
- Calculate reward at each turn: Accuracy(t) - Accuracy(t-1) + shaping
- Update actor-critic using DDPG

**Checkpointing:**
- Save every 10 episodes
- Milestone checkpoints every 100 episodes

In [ ]:
from tqdm import tqdm
import json
import torch
import numpy as np

# Training state
episode_rewards = []
episode_acc = []
train_losses = []
start_episode = 0

# Checkpoint paths
CHECKPOINT_INTERVAL = 10  # Save model checkpoint every N episodes
LOG_INTERVAL = 50  # Print summary every N episodes
STATS_FILE = CHECKPOINT_DIR / 'training_stats.json'
PROGRESS_FILE = CHECKPOINT_DIR / 'training_progress.csv'

def save_checkpoint(path, episode, rewards, acc, losses):
    """Save checkpoint with all state including target networks."""
    checkpoint = {
        'actor': casper_agent.rl_agent.actor.state_dict(),
        'critic': casper_agent.rl_agent.critic.state_dict(),
        'actor_target': casper_agent.rl_agent.actor_target.state_dict(),
        'critic_target': casper_agent.rl_agent.critic_target.state_dict(),
        'episode': episode,
        'rewards': rewards,
        'acc': acc,
        'losses': losses,
        'state_encoder_type': casper_agent.state_encoder_type,
        'reward_type': 'concept_accuracy',
        'reward_stats': {
            'mean': casper_agent.rl_agent.reward_mean,
            'std': casper_agent.rl_agent.reward_std,
            'count': casper_agent.rl_agent.reward_count
        },
        'noise_scale': casper_agent.rl_agent.noise_scale
    }
    if casper_agent.rl_agent.state_encoder is not None:
        checkpoint['state_encoder'] = casper_agent.rl_agent.state_encoder.state_dict()
    torch.save(checkpoint, path)

def save_training_stats(path, rewards, acc, losses):
    """Save training statistics to JSON."""
    stats = {
        'num_episodes': len(rewards),
        'reward_type': 'concept_accuracy',
        'rewards': rewards,
        'acc': acc,
        'losses': losses,
        'state_encoder_type': casper_agent.state_encoder_type
    }
    if len(rewards) >= 10:
        stats['recent_reward_mean'] = float(np.mean(rewards[-10:]))
        stats['recent_acc_mean'] = float(np.mean(acc[-10:]))
    with open(path, 'w') as f:
        json.dump(stats, f, indent=2)

def save_progress_csv(path, rewards, acc, losses):
    """Save episode-level progress as CSV (called every episode)."""
    with open(path, 'w') as f:
        f.write('episode,reward,acc,loss,reward_ma10,acc_ma10\n')
        for i in range(len(rewards)):
            r_ma = np.mean(rewards[max(0,i-9):i+1])
            a_ma = np.mean(acc[max(0,i-9):i+1])
            l = losses[i] if i < len(losses) else 0
            f.write(f'{i+1},{rewards[i]:.4f},{acc[i]:.4f},{l:.4f},{r_ma:.4f},{a_ma:.4f}\n')

def print_progress_summary(ep, rewards, acc, losses):
    """Print training progress summary."""
    n = len(rewards)
    if n < 10:
        return
    
    recent_r = np.mean(rewards[-10:])
    recent_a = np.mean(acc[-10:])
    recent_l = np.mean(losses[-10:]) if losses else 0
    early_r = np.mean(rewards[:10])
    early_a = np.mean(acc[:10])
    r_delta = recent_r - early_r
    a_delta = recent_a - early_a
    
    print(f"\n{'='*60}")
    print(f"PROGRESS SUMMARY (Episode {ep+1})")
    print(f"{'='*60}")
    print(f"                    Early (1-10)    Recent (last 10)    Delta")
    print(f"  Reward:           {early_r:+.4f}          {recent_r:+.4f}          {r_delta:+.4f}")
    print(f"  Accuracy:         {early_a:.4f}           {recent_a:.4f}           {a_delta:+.4f}")
    print(f"  Actor Loss:       -               {recent_l:.4f}")
    print(f"{'='*60}\n")

def load_checkpoint(path):
    """Load checkpoint with encoder type validation and target networks."""
    checkpoint = torch.load(path)
    checkpoint_encoder = checkpoint.get('state_encoder_type', 'sbert')
    current_encoder = casper_agent.state_encoder_type
    
    if checkpoint_encoder != current_encoder:
        print(f"Skipping incompatible checkpoint: {checkpoint_encoder} != {current_encoder}")
        return None
    
    casper_agent.rl_agent.actor.load_state_dict(checkpoint['actor'])
    casper_agent.rl_agent.critic.load_state_dict(checkpoint['critic'])
    
    if 'actor_target' in checkpoint:
        casper_agent.rl_agent.actor_target.load_state_dict(checkpoint['actor_target'])
    else:
        casper_agent.rl_agent.actor_target.load_state_dict(checkpoint['actor'])
        
    if 'critic_target' in checkpoint:
        casper_agent.rl_agent.critic_target.load_state_dict(checkpoint['critic_target'])
    else:
        casper_agent.rl_agent.critic_target.load_state_dict(checkpoint['critic'])
    
    if 'state_encoder' in checkpoint and casper_agent.rl_agent.state_encoder is not None:
        casper_agent.rl_agent.state_encoder.load_state_dict(checkpoint['state_encoder'])
    
    if 'reward_stats' in checkpoint:
        stats = checkpoint['reward_stats']
        casper_agent.rl_agent.reward_mean = stats.get('mean', 0.0)
        casper_agent.rl_agent.reward_std = stats.get('std', 1.0)
        casper_agent.rl_agent.reward_count = stats.get('count', 0)
    
    if 'noise_scale' in checkpoint:
        casper_agent.rl_agent.noise_scale = checkpoint['noise_scale']
    
    return checkpoint

def store_episode_experiences(agent, result):
    """Store experiences from episode for RL training."""
    states = result.get('states', [])
    embeddings = result.get('embeddings', [])
    rewards = result.get('rewards', [])
    
    if len(states) < 2 or len(embeddings) < 2:
        return
    
    for i in range(len(embeddings)):
        if i >= len(rewards):
            break
        state = states[i]
        embedding = embeddings[i]
        reward = rewards[i]
        next_state = states[i + 1] if i + 1 < len(states) else states[i]
        done = (i == len(embeddings) - 1)
        agent.rl_agent.store_experience(state, embedding, reward, next_state, done)

# Try to load latest compatible checkpoint
for ckpt_path in sorted(CHECKPOINT_DIR.glob('rl_episode_*.pt'), reverse=True):
    checkpoint = load_checkpoint(ckpt_path)
    if checkpoint:
        start_episode = checkpoint['episode']
        episode_rewards = checkpoint.get('rewards', [])
        # Support both old 'ndcg' key and new 'acc' key
        episode_acc = checkpoint.get('acc', checkpoint.get('ndcg', []))
        train_losses = checkpoint.get('losses', [])
        print(f"Resumed from {ckpt_path} (episode {start_episode})")
        break
else:
    print(f"Starting fresh (encoder: {casper_agent.state_encoder_type})")
    # Clear old logs
    for log_file in [CHECKPOINT_DIR / 'training_conversations.jsonl', 
                     CHECKPOINT_DIR / 'training_detailed.jsonl',
                     PROGRESS_FILE]:
        if log_file.exists():
            log_file.unlink()
            print(f"  Cleared {log_file.name}")

# Open log files
conv_log = open(CHECKPOINT_DIR / 'training_conversations.jsonl', 'a', encoding='utf-8')
detailed_log = open(CHECKPOINT_DIR / 'training_detailed.jsonl', 'a', encoding='utf-8')

print(f"\nRL Training: {start_episode} -> {RL_EPISODES} episodes")
print(f"Reward signal: Concept Model Accuracy")
print(f"Model checkpoint every {CHECKPOINT_INTERVAL} episodes")
print(f"Progress summary every {LOG_INTERVAL} episodes")
print(f"CSV saved every episode")
print(f"State encoder: {casper_agent.state_encoder_type}")
print(f"\nMonitor progress:")
print(f"  - Progress CSV: {PROGRESS_FILE}")
print(f"  - Stats JSON: {STATS_FILE}")

pbar = tqdm(range(start_episode, RL_EPISODES), desc="Training", unit="ep", 
            initial=start_episode, total=RL_EPISODES)

try:
    for ep in pbar:
        # Sample user and run episode
        profile = user_sim.sample_user()
        result = episode_runner.run_episode(casper_agent, profile)
        user_id = result['user_ground_truth'].get('user_id', 'unknown')
        
        # Store experiences with CORRECT state encoding
        store_episode_experiences(casper_agent, result)
        
        # Train RL agent
        train_metrics = casper_agent.rl_agent.train_step()
        actor_loss = train_metrics.get('actor_loss', 0) if train_metrics else 0
        avg_q = train_metrics.get('avg_q', 0) if train_metrics else 0
        
        # Track metrics
        # NOTE: result['final_ndcg'] contains accuracy (EpisodeRunner naming)
        episode_rewards.append(result['total_reward'])
        episode_acc.append(result['final_ndcg'])
        train_losses.append(actor_loss)
        
        # Log conversation summary
        conv_log.write(json.dumps({
            'episode': ep + 1,
            'user_id': user_id,
            'num_eval_targets': result['user_ground_truth'].get('num_eval_targets'),
            'reward': result['total_reward'],
            'accuracy': result['final_ndcg'],
            'loss': actor_loss,
            'avg_q': avg_q,
            'preferences': casper_agent.discovered_preferences
        }) + '\n')
        conv_log.flush()
        
        # Detailed turn log (JSON)
        detailed_log.write(json.dumps({
            'episode': ep + 1,
            'user_id': user_id,
            'num_eval_targets': result['user_ground_truth'].get('num_eval_targets'),
            'turns': result['turn_logs'],
            'metrics': {'reward': result['total_reward'], 'accuracy': result['final_ndcg'], 
                       'loss': actor_loss, 'avg_q': avg_q}
        }) + '\n')
        detailed_log.flush()
        
        # Update progress bar
        pbar.set_postfix({
            'r': f'{result["total_reward"]:.3f}',
            'acc': f'{result["final_ndcg"]:.3f}',
            'Q': f'{avg_q:.2f}'
        })
        
        # Save CSV every episode for real-time monitoring
        save_progress_csv(PROGRESS_FILE, episode_rewards, episode_acc, train_losses)
        
        # Save model checkpoint periodically
        if (ep + 1) % CHECKPOINT_INTERVAL == 0:
            ckpt_path = CHECKPOINT_DIR / f'rl_episode_{ep+1}.pt'
            save_checkpoint(ckpt_path, ep + 1, episode_rewards, episode_acc, train_losses)
            save_training_stats(STATS_FILE, episode_rewards, episode_acc, train_losses)
        
        # Print progress summary periodically
        if (ep + 1) % LOG_INTERVAL == 0:
            print_progress_summary(ep, episode_rewards, episode_acc, train_losses)

except KeyboardInterrupt:
    print("\nInterrupted - saving checkpoint...")
    save_checkpoint(CHECKPOINT_DIR / f'rl_interrupted_{ep+1}.pt', ep + 1, 
                   episode_rewards, episode_acc, train_losses)
finally:
    conv_log.close()
    detailed_log.close()
    save_training_stats(STATS_FILE, episode_rewards, episode_acc, train_losses)
    save_progress_csv(PROGRESS_FILE, episode_rewards, episode_acc, train_losses)

print(f"\nDone! {len(episode_rewards)} episodes")
if episode_rewards:
    print(f"Mean reward (last 10): {np.mean(episode_rewards[-10:]):.3f}")
    print(f"Mean accuracy (last 10): {np.mean(episode_acc[-10:]):.3f}")

## 8. Load Best Checkpoint for Evaluation

In [ ]:
# Load most recent checkpoint for evaluation
# Priority: rl_interrupted_*.pt > rl_episode_*.pt

checkpoints_to_try = [
    *sorted(CHECKPOINT_DIR.glob('rl_interrupted_*.pt'), reverse=True),
    *sorted(CHECKPOINT_DIR.glob('rl_episode_*.pt'), reverse=True)
]

loaded = False
for checkpoint_path in checkpoints_to_try:
    if checkpoint_path.exists():
        print(f"Loading checkpoint for evaluation: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        casper_agent.rl_agent.actor.load_state_dict(checkpoint['actor'])
        casper_agent.rl_agent.critic.load_state_dict(checkpoint['critic'])
        
        if 'actor_target' in checkpoint:
            casper_agent.rl_agent.actor_target.load_state_dict(checkpoint['actor_target'])
        if 'critic_target' in checkpoint:
            casper_agent.rl_agent.critic_target.load_state_dict(checkpoint['critic_target'])
        
        if 'state_encoder' in checkpoint and casper_agent.rl_agent.state_encoder is not None:
            casper_agent.rl_agent.state_encoder.load_state_dict(checkpoint['state_encoder'])
            print(f"State encoder loaded (type: {checkpoint.get('state_encoder_type', 'unknown')})")
        
        print(f"Reward type: {checkpoint.get('reward_type', 'unknown')}")
        print(f"Model trained for {checkpoint['episode']} episodes")
        
        acc_list = checkpoint.get('acc', checkpoint.get('ndcg', []))
        if acc_list:
            recent_acc = np.mean(acc_list[-min(10, len(acc_list)):])
            print(f"Training accuracy (last 10 eps): {recent_acc:.3f}")
        
        loaded = True
        break

if not loaded:
    print("WARNING: No checkpoints found - using untrained agent!")
    print("This will produce poor results.")

In [ ]:
import random

print("Evaluating agents on test users...")

N_EVAL_USERS = min(50, len(test_users))
eval_users = random.sample(test_users, N_EVAL_USERS)

def evaluate_agent(agent, users, runner):
    """Evaluate agent on users using episode runner."""
    acc_scores = []
    rewards = []
    
    for user in tqdm(users, desc=f"Evaluating {agent.__class__.__name__}", unit="user"):
        try:
            result = runner.run_episode(agent, user, verbose=False)
            # result['final_ndcg'] contains accuracy (EpisodeRunner naming)
            acc_scores.append(result['final_ndcg'])
            rewards.append(result['total_reward'])
        except Exception as e:
            print(f"Error: {e}")
    
    return {
        'acc_mean': np.mean(acc_scores) if acc_scores else 0,
        'acc_std': np.std(acc_scores) if acc_scores else 0,
        'reward_mean': np.mean(rewards) if rewards else 0,
        'reward_std': np.std(rewards) if rewards else 0,
        'n_users': len(acc_scores)
    }

# Evaluate all agents
casper_results = evaluate_agent(casper_agent, eval_users, episode_runner)
print(f"CASPER Accuracy: {casper_results['acc_mean']:.3f} +/- {casper_results['acc_std']:.3f}")

llm_results = evaluate_agent(llm_agent, eval_users, episode_runner)
print(f"LLM Accuracy: {llm_results['acc_mean']:.3f} +/- {llm_results['acc_std']:.3f}")

random_results = evaluate_agent(random_agent, eval_users, episode_runner)
print(f"Random Accuracy: {random_results['acc_mean']:.3f} +/- {random_results['acc_std']:.3f}")

## 9. Results Comparison

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    'Agent': ['CASPER (ours)', 'PureLLMAgent', 'RandomAgent'],
    'Accuracy': [
        f"{casper_results['acc_mean']:.3f} +/- {casper_results['acc_std']:.3f}",
        f"{llm_results['acc_mean']:.3f} +/- {llm_results['acc_std']:.3f}",
        f"{random_results['acc_mean']:.3f} +/- {random_results['acc_std']:.3f}"
    ],
    'Total Reward': [
        f"{casper_results['reward_mean']:.3f} +/- {casper_results['reward_std']:.3f}",
        f"{llm_results['reward_mean']:.3f} +/- {llm_results['reward_std']:.3f}",
        f"{random_results['reward_mean']:.3f} +/- {random_results['reward_std']:.3f}"
    ]
})

print("\n" + "="*60)
print("RESULTS (Concept Model Accuracy)")
print("="*60 + "\n")
print(results_df.to_string(index=False))
print("\n" + "="*60)

# Save results
results_path = CHECKPOINT_DIR / 'results.csv'
results_df.to_csv(results_path, index=False)
print(f"\nResults saved: {results_path}")

## 10. Training Visualization

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('RL Training: Rewards', 'RL Training: Concept Model Accuracy')
)

# Rewards plot
fig.add_trace(
    go.Scatter(y=episode_rewards, mode='lines', name='Episode', opacity=0.3, line=dict(color='blue')),
    row=1, col=1
)

# Moving average
window = 50
if len(episode_rewards) >= window:
    smooth_rewards = pd.Series(episode_rewards).rolling(window).mean()
    fig.add_trace(
        go.Scatter(y=smooth_rewards, mode='lines', name=f'{window}-episode MA', line=dict(color='blue', width=2)),
        row=1, col=1
    )

# Accuracy plot
fig.add_trace(
    go.Scatter(y=episode_acc, mode='lines', name='Episode', opacity=0.3, line=dict(color='red'), showlegend=False),
    row=1, col=2
)

if len(episode_acc) >= window:
    smooth_acc = pd.Series(episode_acc).rolling(window).mean()
    fig.add_trace(
        go.Scatter(y=smooth_acc, mode='lines', name=f'{window}-episode MA', line=dict(color='red', width=2), showlegend=False),
        row=1, col=2
    )

fig.update_xaxes(title_text="Episode", row=1, col=1)
fig.update_xaxes(title_text="Episode", row=1, col=2)
fig.update_yaxes(title_text="Total Reward", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

# Save plot
plot_path = CHECKPOINT_DIR / 'training_progress.html'
fig.write_html(str(plot_path))
print(f"Plot saved: {plot_path}")

## 11. Save Final Checkpoint

In [ ]:
final_checkpoint = CHECKPOINT_DIR / 'casper_final.pt'

checkpoint_data = {
    'actor': casper_agent.rl_agent.actor.state_dict(),
    'critic': casper_agent.rl_agent.critic.state_dict(),
    'actor_target': casper_agent.rl_agent.actor_target.state_dict(),
    'critic_target': casper_agent.rl_agent.critic_target.state_dict(),
    'num_episodes': RL_EPISODES,
    'episode_rewards': episode_rewards,
    'episode_acc': episode_acc,
    'state_encoder_type': casper_agent.state_encoder_type,
    'reward_type': 'concept_accuracy',
    'config': config
}

# Include state encoder if using LSTM mode
if casper_agent.rl_agent.state_encoder is not None:
    checkpoint_data['state_encoder'] = casper_agent.rl_agent.state_encoder.state_dict()

torch.save(checkpoint_data, final_checkpoint)
print(f"Final checkpoint saved: {final_checkpoint}")
print(f"\nTraining complete!")
print(f"  Total episodes: {RL_EPISODES}")
print(f"  State encoder: {casper_agent.state_encoder_type}")
print(f"  Reward signal: Concept Model Accuracy")
print(f"  Final accuracy: {np.mean(episode_acc[-100:]):.3f}" if len(episode_acc) >= 100 else f"  Final accuracy: {np.mean(episode_acc):.3f}")